In [1]:
%pip install -U -q pip setuptools wheel setuptools-scm
%pip uninstall -y -q diffusers
%pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 seqeval==1.2.2
%pip install -q optimum optimum-onnx onnx==1.16.2 onnxruntime==1.19.2 onnxscript
print("ok")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 38.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.1/109.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... 

In [2]:
import json, sys, importlib, random
import numpy as np, torch

SEED = 42
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

ENTITY_TYPES = ["ITEM", "QTY", "UNIT", "VARIANT", "ANAPHORIC"]
LABELS = ["O"] + [f"{p}-{e}" for e in ENTITY_TYPES for p in ("B", "I")]
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

DATA = "/kaggle/input/datasets/sebastianabe/nyatet-order-train"
B5 = "/kaggle/input/datasets/sebastianabe/nyatet-order-train-b5"

sys.path.insert(0, DATA)
import generate_data; importlib.reload(generate_data)

train_rows = generate_data.generate_dataset(n_orders=8000)
generate_data.validate(train_rows)
eval_rows = json.load(open(f"{B5}/eval_annotated.json", encoding="utf-8"))

print(f"{len(LABELS)} labels | train {len(train_rows)} | eval {len(eval_rows)}")

11 labels | train 10800 | eval 81


In [3]:
from transformers import BertTokenizerFast
from datasets import Dataset

TEACHER_NAME = "indobenchmark/indobert-base-p2"
tokenizer = BertTokenizerFast.from_pretrained(TEACHER_NAME)

def encode(row):
    enc = tokenizer(row["text"], return_offsets_mapping=True, truncation=True, max_length=96)
    offsets = enc.pop("offset_mapping")
    tags = [None if a == b else "O" for a, b in offsets]
    for span in row["spans"]:
        entered = False
        for i, (a, b) in enumerate(offsets):
            if a == b: continue
            if a >= span["start"] and b <= span["end"]:
                tags[i] = f'{"B" if not entered else "I"}-{span["type"]}'
                entered = True
    enc["labels"] = [-100 if t is None else LABEL2ID[t] for t in tags]
    return enc

train_ds = Dataset.from_list([encode(r) for r in train_rows]).train_test_split(test_size=0.05, seed=SEED)
eval_ds  = Dataset.from_list([encode(r) for r in eval_rows])
print(train_ds)

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10260
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 540
    })
})


In [4]:
from seqeval.metrics import f1_score, classification_report
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)

def metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    tp = [[ID2LABEL[a] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    tl = [[ID2LABEL[b] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    return {"f1": f1_score(tl, tp)}

collator = DataCollatorForTokenClassification(tokenizer)

teacher = AutoModelForTokenClassification.from_pretrained(
    TEACHER_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)

teacher_trainer = Trainer(
    model=teacher,
    args=TrainingArguments(
        output_dir="./checkpoints/teacher",
        num_train_epochs=8,              
        per_device_train_batch_size=32,  
        per_device_eval_batch_size=64,
        learning_rate=5e-5,              
        warmup_ratio=0.1,                
        weight_decay=0.01,               
        eval_strategy="epoch", save_strategy="no",
        logging_steps=50, report_to="none", fp16=True, seed=SEED),
    train_dataset=train_ds["train"], eval_dataset=train_ds["test"],
    data_collator=collator, compute_metrics=metrics)

teacher_trainer.train()
teacher_trainer.save_model("./checkpoints/teacher")

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,0.006600,0.000369,0.998737
2,0.002400,0.000000,1.000000
3,0.000000,0.000000,1.000000
4,0.000000,0.000000,1.000000
5,0.000000,0.000000,1.000000
6,0.000600,0.000000,1.000000
7,0.000000,0.000000,1.000000
8,0.000000,0.000000,1.000000


In [5]:
def report(trainer, dataset, title):
    preds, labels, _ = trainer.predict(dataset)
    preds = np.argmax(preds, axis=2)
    tp = [[ID2LABEL[a] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    tl = [[ID2LABEL[b] for a, b in zip(pr, lb) if b != -100] for pr, lb in zip(preds, labels)]
    f1 = f1_score(tl, tp)
    print(f"=== {title} ===\nF1 {f1:.4f}\n")
    print(classification_report(tl, tp, digits=3))
    return f1

teacher_f1 = report(teacher_trainer, eval_ds, f"TEACHER on real held-out (n={len(eval_rows)})")

=== TEACHER on real held-out (n=81) ===
F1 0.8378

              precision    recall  f1-score   support

   ANAPHORIC      0.125     1.000     0.222         2
        ITEM      0.857     0.857     0.857        21
         QTY      0.943     0.943     0.943        35
        UNIT      0.778     1.000     0.875        21
     VARIANT      0.760     1.000     0.864        19

   micro avg      0.750     0.949     0.838        98
   macro avg      0.693     0.960     0.752        98
weighted avg      0.837     0.949     0.880        98



In [6]:
from optimum.onnxruntime import ORTModelForTokenClassification
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
from pathlib import Path
import shutil, time, numpy as np, onnxruntime as ort
from seqeval.metrics import f1_score

def eval_onnx(path):
    sess = ort.InferenceSession(path)
    names = [i.name for i in sess.get_inputs()]
    tp, tl = [], []
    for row in eval_rows:
        enc = tokenizer(row["text"], truncation=True, max_length=96, return_tensors="np")
        feed = {n: enc[n].astype(np.int64) for n in names}
        pred = sess.run(None, feed)[0][0].argmax(-1)
        for p, g in zip(pred, encode(row)["labels"]):
            if g != -100:
                tp.append(ID2LABEL[int(p)]); tl.append(ID2LABEL[int(g)])
    return f1_score([tl], [tp])

def bench(path, n=100):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    s = ort.InferenceSession(path, so)
    enc = tokenizer("bu risol mentah 20 biji, jam 7 pagi diambil", return_tensors="np")
    feed = {i.name: enc[i.name].astype(np.int64) for i in s.get_inputs()}
    for _ in range(20): s.run(None, feed)
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); s.run(None, feed); ts.append((time.perf_counter()-t0)*1000)
    ts.sort()
    return ts[n//2], ts[int(n*0.95)-1]

m = ORTModelForTokenClassification.from_pretrained("./checkpoints/teacher", export=True)
m.save_pretrained("./t_fp32"); tokenizer.save_pretrained("./t_fp32")

quant_pre_process(input_model_path="t_fp32/model.onnx",
                  output_model_path="t_fp32/model_pre.onnx", skip_symbolic_shape=True)

for name, ops in (("t_int8_v2", ["MatMul"]), ("t_int8_v3", ["MatMul", "Gather"])):
    Path(name).mkdir(parents=True, exist_ok=True)
    quantize_dynamic("t_fp32/model_pre.onnx", f"{name}/model.onnx",
                     weight_type=QuantType.QInt8, op_types_to_quantize=ops,
                     extra_options={"MatMulConstBOnly": False})
    tokenizer.save_pretrained(name)
    shutil.copy("./checkpoints/teacher/config.json", f"{name}/config.json")

print(f"{'variant':<20} {'F1':>7} {'size MB':>9} {'median':>8} {'p95':>7}")
for name, path in {"fp32": "t_fp32/model.onnx",
                   "int8 v2": "t_int8_v2/model.onnx",
                   "int8 v3": "t_int8_v3/model.onnx"}.items():
    if not Path(path).exists(): continue
    med, p95 = bench(path)
    print(f"{name:<20} {eval_onnx(path):>7.4f} {Path(path).stat().st_size/1024**2:>9.2f} "
          f"{med:>7.1f}ms {p95:>6.1f}ms")

variant                   F1   size MB   median     p95
fp32                  0.8378    472.73    48.1ms   51.9ms
int8 v2               0.8378    229.79    24.8ms   25.9ms
int8 v3               0.8378    118.80    24.8ms   25.6ms


In [7]:
from transformers import BertConfig, BertForTokenClassification

t_cfg = teacher.config
s_cfg = BertConfig(
    vocab_size=t_cfg.vocab_size,
    hidden_size=384,
    num_hidden_layers=4,
    num_attention_heads=6,
    intermediate_size=1536,
    max_position_embeddings=t_cfg.max_position_embeddings,
    type_vocab_size=t_cfg.type_vocab_size,
    num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID)

student = BertForTokenClassification(s_cfg)

n_t = sum(p.numel() for p in teacher.parameters())
n_s = sum(p.numel() for p in student.parameters())
print(f"teacher {n_t/1e6:.1f}M  student {n_s/1e6:.1f}M  ratio {n_t/n_s:.1f}x")
print(f"teacher: {t_cfg.num_hidden_layers}L x {t_cfg.hidden_size}h")
print(f"student: {s_cfg.num_hidden_layers}L x {s_cfg.hidden_size}h")

teacher 123.9M  student 26.5M  ratio 4.7x
teacher: 12L x 768h
student: 4L x 384h


In [8]:
import torch.nn as nn

with torch.no_grad():
    t_emb = teacher.bert.embeddings
    s_emb = student.bert.embeddings
    s_emb.word_embeddings.weight.copy_(t_emb.word_embeddings.weight[:, :384])
    s_emb.position_embeddings.weight.copy_(t_emb.position_embeddings.weight[:, :384])
    s_emb.token_type_embeddings.weight.copy_(t_emb.token_type_embeddings.weight[:, :384])

print("embeddings seeded from teacher (first 384 dims)")

embeddings seeded from teacher (first 384 dims)


In [9]:
import torch.nn.functional as F

ALPHA = 0.5  
TEMP  = 3.0

class DistillTrainer(Trainer):
    def __init__(self, teacher_model=None, **kw):
        super().__init__(**kw)
        self.teacher = teacher_model
        self.teacher.eval()
        for p in self.teacher.parameters():
            p.requires_grad = False

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs["labels"]
        outputs = model(**inputs)
        s_logits = outputs.logits

        with torch.no_grad():
            self.teacher.to(s_logits.device)
            t_logits = self.teacher(**{k: v for k, v in inputs.items() if k != "labels"}).logits

        ce = F.cross_entropy(s_logits.view(-1, len(LABELS)), labels.view(-1), ignore_index=-100)

        mask = labels.view(-1) != -100
        s_flat = s_logits.view(-1, len(LABELS))[mask]
        t_flat = t_logits.view(-1, len(LABELS))[mask]
        kd = F.kl_div(
            F.log_softmax(s_flat / TEMP, dim=-1),
            F.softmax(t_flat / TEMP, dim=-1),
            reduction="batchmean") * (TEMP ** 2)

        loss = ALPHA * ce + (1 - ALPHA) * kd
        return (loss, outputs) if return_outputs else loss

print(f"alpha={ALPHA}  temperature={TEMP}")

alpha=0.5  temperature=3.0


In [10]:
student_trainer = DistillTrainer(
    teacher_model=teacher,
    model=student,
    args=TrainingArguments(
        output_dir="./checkpoints/student", num_train_epochs=15,
        per_device_train_batch_size=32, per_device_eval_batch_size=64,
        learning_rate=1e-4,          
        warmup_ratio=0.1,
        eval_strategy="epoch", save_strategy="no",
        logging_steps=50, report_to="none", fp16=True, seed=SEED),
    train_dataset=train_ds["train"], eval_dataset=train_ds["test"],
    data_collator=collator, compute_metrics=metrics)

student_trainer.train()
student_trainer.save_model("./checkpoints/student")
tokenizer.save_pretrained("./checkpoints/student")

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1
1,2.214000,1.243844,0.917774
2,0.279200,0.120483,0.997055
3,0.052200,0.017763,0.998737
4,0.024400,0.027137,0.999579
5,0.024700,0.012255,1.000000
6,0.014600,0.007059,1.000000
7,0.013600,0.012945,1.000000
8,0.009400,0.012138,0.999579
9,0.006200,0.000478,1.000000
10,0.005500,0.005095,1.000000


('./checkpoints/student/tokenizer_config.json',
 './checkpoints/student/special_tokens_map.json',
 './checkpoints/student/vocab.txt',
 './checkpoints/student/added_tokens.json',
 './checkpoints/student/tokenizer.json')

In [11]:
student_f1 = report(student_trainer, eval_ds, f"STUDENT on real held-out (n={len(eval_rows)})")

print(f"\n{'':<24} {'F1':>8}")
print(f"{'O1 (lite, fine-tuned)':<24} {0.8365:>8.4f}")
print(f"{'O2 teacher (base)':<24} {teacher_f1:>8.4f}")
print(f"{'O2 student (distilled)':<24} {student_f1:>8.4f}")
print(f"\nstudent vs O1: {student_f1 - 0.9022:+.4f}")
print("Differences under ~0.02 are inside the resolution limit at n=81.")

=== STUDENT on real held-out (n=81) ===
F1 0.6917

              precision    recall  f1-score   support

   ANAPHORIC      0.083     1.000     0.154         2
        ITEM      0.654     0.810     0.723        21
         QTY      0.686     1.000     0.814        35
        UNIT      0.467     1.000     0.636        21
     VARIANT      0.773     0.895     0.829        19

   micro avg      0.548     0.939     0.692        98
   macro avg      0.533     0.941     0.631        98
weighted avg      0.637     0.939     0.746        98


                               F1
O1 (lite, fine-tuned)      0.8365
O2 teacher (base)          0.8378
O2 student (distilled)     0.6917

student vs O1: -0.2105
Differences under ~0.02 are inside the resolution limit at n=81.


In [12]:
from optimum.onnxruntime import ORTModelForTokenClassification
from onnxruntime.quantization import quantize_dynamic, QuantType
from onnxruntime.quantization.shape_inference import quant_pre_process
from pathlib import Path
import shutil

m = ORTModelForTokenClassification.from_pretrained("./checkpoints/student", export=True)
m.save_pretrained("./s_fp32"); tokenizer.save_pretrained("./s_fp32")

quant_pre_process(input_model_path="s_fp32/model.onnx",
                  output_model_path="s_fp32/model_pre.onnx", skip_symbolic_shape=True)

for name, ops in (("s_int8_v2", ["MatMul"]), ("s_int8_v3", ["MatMul", "Gather"])):
    Path(name).mkdir(parents=True, exist_ok=True)
    quantize_dynamic("s_fp32/model_pre.onnx", f"{name}/model.onnx",
                     weight_type=QuantType.QInt8, op_types_to_quantize=ops,
                     extra_options={"MatMulConstBOnly": False})
    tokenizer.save_pretrained(name)
    shutil.copy("./checkpoints/student/config.json", f"{name}/config.json")

for d in ("s_fp32", "s_int8_v2", "s_int8_v3"):
    for f in sorted(Path(d).glob("*.onnx")):
        print(f"{str(f):<34} {f.stat().st_size/1024**2:6.2f} MB")

s_fp32/model.onnx                  101.18 MB
s_fp32/model_pre.onnx              101.16 MB
s_int8_v2/model.onnx                80.94 MB
s_int8_v3/model.onnx                25.45 MB


In [13]:
import time, onnxruntime as ort

def eval_onnx(path, report_=False):
    sess = ort.InferenceSession(path)
    names = [i.name for i in sess.get_inputs()]
    tp, tl = [], []
    for row in eval_rows:
        enc = tokenizer(row["text"], truncation=True, max_length=96, return_tensors="np")
        feed = {n: enc[n].astype(np.int64) for n in names}
        pred = sess.run(None, feed)[0][0].argmax(-1)
        for p, g in zip(pred, encode(row)["labels"]):
            if g != -100:
                tp.append(ID2LABEL[int(p)]); tl.append(ID2LABEL[int(g)])
    if report_: print(classification_report([tl], [tp], digits=3))
    return f1_score([tl], [tp])

def bench(path, n=100):
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    s = ort.InferenceSession(path, so)
    enc = tokenizer("bu risol mentah 20 biji, jam 7 pagi diambil", return_tensors="np")
    feed = {i.name: enc[i.name].astype(np.int64) for i in s.get_inputs()}
    for _ in range(20): s.run(None, feed)
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); s.run(None, feed); ts.append((time.perf_counter()-t0)*1000)
    ts.sort()
    return ts[n//2], ts[int(n*0.95)-1]

print(f"{'variant':<20} {'F1':>7} {'size MB':>9} {'median':>8} {'p95':>7}")
for name, path in {"fp32": "s_fp32/model.onnx",
                   "int8 v2": "s_int8_v2/model.onnx",
                   "int8 v3": "s_int8_v3/model.onnx"}.items():
    if not Path(path).exists(): continue
    med, p95 = bench(path)
    print(f"{name:<20} {eval_onnx(path):>7.4f} {Path(path).stat().st_size/1024**2:>9.2f} "
          f"{med:>7.1f}ms {p95:>6.1f}ms")

print("\n--- per-class, int8 v3 ---")
eval_onnx("s_int8_v3/model.onnx", report_=True)

variant                   F1   size MB   median     p95
fp32                  0.6917    101.18     3.7ms    4.1ms
int8 v2               0.6917     80.94     2.5ms    2.7ms
int8 v3               0.6917     25.45     2.5ms    2.7ms

--- per-class, int8 v3 ---
              precision    recall  f1-score   support

   ANAPHORIC      0.083     1.000     0.154         2
        ITEM      0.654     0.810     0.723        21
         QTY      0.686     1.000     0.814        35
        UNIT      0.467     1.000     0.636        21
     VARIANT      0.773     0.895     0.829        19

   micro avg      0.548     0.939     0.692        98
   macro avg      0.533     0.941     0.631        98
weighted avg      0.637     0.939     0.746        98



np.float64(0.6917293233082707)

In [14]:
import zipfile
from pathlib import Path

for d in ["s_fp32", "s_int8_v2", "s_int8_v3"]:
    p = Path(d)
    if not p.exists(): continue
    out = Path(f"/kaggle/working/{d}.zip")
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(p.rglob("*")):
            if f.is_file() and f.name != "model_pre.onnx":
                z.write(f, f.name)
    print(f"{out.name:<20} {out.stat().st_size/1024**2:6.2f} MB")

s_fp32.zip            94.19 MB
s_int8_v2.zip         74.89 MB
s_int8_v3.zip         20.37 MB


In [15]:
import onnxruntime as ort, numpy as np

MODELS = {
    "teacher (base-p2)": "t_int8_v3/model.onnx",
    "student (4-layer)": "s_int8_v3/model.onnx",
}

n_neg = sum(1 for r in eval_rows if not r["spans"])

for label, path in MODELS.items():
    sess = ort.InferenceSession(path)
    names = [i.name for i in sess.get_inputs()]
    fp = 0
    for row in eval_rows:
        if row["spans"]: continue
        enc = tokenizer(row["text"], truncation=True, max_length=96, return_tensors="np")
        feed = {n: enc[n].astype(np.int64) for n in names}
        pred = sess.run(None, feed)[0][0].argmax(-1)
        gold = encode(row)["labels"]
        if any(ID2LABEL[int(p)] != "O" for p, g in zip(pred, gold) if g != -100):
            fp += 1
    print(f"{label:<20} {fp}/{n_neg} ({fp/n_neg*100:.0f}%)")

teacher (base-p2)    5/41 (12%)
student (4-layer)    16/41 (39%)
